In [ ]:
import control as ctrl
import numpy as np
import sympy as sp

from lib.plots import plot_or_show, plt


def plot_bode(G, name):
    mag, phase, omega = ctrl.frequency_response(G, np.logspace(-1, 4, 1000))
    phase = np.unwrap(phase)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    # Magnitude (dB)
    ax1.semilogx(omega, 20 * np.log10(mag))
    ax1.grid(True, which="both")
    ax1.set_ylabel("Magnitude / dB")

    # Fase (graus)
    ax2.semilogx(omega, np.degrees(phase))
    ax2.grid(True, which="both")
    ax2.set_ylabel("Fase / (graus)")
    ax2.set_xlabel("Frequência / (rad$\\cdot$h$^{-1}$)")

    plot_or_show(f"bode/{name}")


In [2]:
import hickle as hkl

from lib.utils import G_sp_to_ctrl, input_names, output_names

G_matrix = hkl.load("../outputs/G_simplified.hkl")

for i, output_name in enumerate(output_names):
    for j, input_name in enumerate(input_names):
        G_sym = sp.simplify(G_matrix[i, j])

        G_name = f"{output_name}_{input_name}"

        G = G_sp_to_ctrl(G_sym)
        plot_bode(G, G_name)


Plot saved to ../figures/bode/T1_Ff1.png
Plot saved to ../figures/bode/T1_Ff2.png
Plot saved to ../figures/bode/T1_FR.png
Plot saved to ../figures/bode/T1_Q1.png
Plot saved to ../figures/bode/T1_Q2.png
Plot saved to ../figures/bode/T1_Q3.png
Plot saved to ../figures/bode/T1_T0.png
Plot saved to ../figures/bode/T2_Ff1.png
Plot saved to ../figures/bode/T2_Ff2.png
Plot saved to ../figures/bode/T2_FR.png
Plot saved to ../figures/bode/T2_Q1.png
Plot saved to ../figures/bode/T2_Q2.png
Plot saved to ../figures/bode/T2_Q3.png
Plot saved to ../figures/bode/T2_T0.png
Plot saved to ../figures/bode/T3_Ff1.png
Plot saved to ../figures/bode/T3_Ff2.png
Plot saved to ../figures/bode/T3_FR.png
Plot saved to ../figures/bode/T3_Q1.png
Plot saved to ../figures/bode/T3_Q2.png
Plot saved to ../figures/bode/T3_Q3.png
Plot saved to ../figures/bode/T3_T0.png
Plot saved to ../figures/bode/xA1_Ff1.png
Plot saved to ../figures/bode/xA1_Ff2.png
Plot saved to ../figures/bode/xA1_FR.png
Plot saved to ../figures/bode

In [3]:
# Função de transferência (3,3)
G_sym = sp.simplify(G_matrix[2, 2])
G_name = f"{output_names[2]}_{input_names[2]}"
G = G_sp_to_ctrl(G_sym)

# ============================
# Polos e zeros
# ============================

poles = ctrl.poles(G)
zeros = ctrl.zeros(G)

print("Polos:")
for p in poles:
    print(f"  {p}")

print("\nZeros:")
for z in zeros:
    print(f"  {z}")

print("\nFrequências dos polos:")
for p in poles:
    if np.isclose(np.imag(p), 0):
        wc = -np.real(p)
        print(f"Polo real: {p:.6g} -> ωp = {wc:.6g} rad/h")
    elif np.imag(p) > 0:
        wn = np.abs(p)
        print(f"Par complexo: {p:.6g} -> ωn = {wn:.6g} rad/h")

print("\nFrequências dos zeros:")
for z in zeros:
    if np.isclose(np.imag(z), 0):
        wz = -np.real(z)
        print(f"Zero real: {z:.6g} -> ωz = {wz:.6g} rad/h")
    elif np.imag(z) > 0:
        wn = np.abs(z)
        print(f"Par complexo: {z:.6g} -> ωn = {wn:.6g} rad/h")

# ============================
# Diagrama de Bode
# ============================

mag, phase, omega = ctrl.frequency_response(G, np.logspace(-1, 4, 1000))
phase = np.unwrap(phase)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Magnitude
ax1.semilogx(omega, 20 * np.log10(mag))
ax1.grid(True, which="both")
ax1.set_ylabel("Magnitude / dB")

# Fase
ax2.semilogx(omega, np.degrees(phase))
ax2.grid(True, which="both")
ax2.set_ylabel("Fase / (graus)")
ax2.set_xlabel(r"Frequência / (rad$\cdot$h$^{-1}$)")

# ===================================================
# Marcação das frequências características
# ===================================================

real_poles = [p for p in poles if np.isclose(np.imag(p), 0)]
complex_poles = [p for p in poles if np.imag(p) > 0]

real_zeros = [z for z in zeros if np.isclose(np.imag(z), 0)]
complex_zeros = [z for z in zeros if np.imag(z) > 0]


def draw_marker(freq, label, color, linestyle):
    for ax in (ax1, ax2):
        ax.axvline(freq, color=color, linestyle=linestyle, linewidth=1.2)

    ax1.text(
        freq,
        -0.02,
        label,
        transform=ax1.get_xaxis_transform(),
        rotation=90,
        ha="center",
        va="top",
        color=color,
        fontsize=12,
        clip_on=False,
    )


# ---------- Polos reais (preto tracejado) ----------
for i, p in enumerate(real_poles):
    freq = abs(np.real(p))

    if len(real_poles) == 1:
        label = r"$w_p$"
    else:
        label = rf"$w_{{p{i + 1}}}$"

    draw_marker(freq, label, "black", "--")

# ---------- Par(es) complexo(s) (preto tracejado) ----------
for p in complex_poles:
    freq = abs(p)
    draw_marker(freq, r"$w_n$", "black", "--")

# ---------- Zeros reais (vermelho contínuo) ----------
for i, z in enumerate(real_zeros):
    freq = abs(np.real(z))

    if len(real_zeros) == 1:
        label = r"$w_z$"
    else:
        label = rf"$w_{{z{i + 1}}}$"

    draw_marker(freq, label, "red", "-")

# ---------- Zeros complexos (vermelho contínuo) ----------
for z in complex_zeros:
    freq = abs(z)
    draw_marker(freq, r"$w_n$", "red", "-")

plot_or_show(f"bode_{G_name}")


Polos:
  (-48.0375021573731+19.61988059704532j)
  (-48.0375021573731-19.61988059704532j)
  (-6.9602900466688435+0j)
  (-3.1793943794393864+0j)

Zeros:
  (13.325775586262882+0j)
  (-23.60684662141628+0j)
  (-9.01348494909729+0j)

Frequências dos polos:
Par complexo: -48.0375+19.6199j -> ωn = 51.8897 rad/h
Polo real: -6.96029+0j -> ωp = 6.96029 rad/h
Polo real: -3.17939+0j -> ωp = 3.17939 rad/h

Frequências dos zeros:
Zero real: 13.3258+0j -> ωz = -13.3258 rad/h
Zero real: -23.6068+0j -> ωz = 23.6068 rad/h
Zero real: -9.01348+0j -> ωz = 9.01348 rad/h
Plot saved to ../figures/bode_T3_FR.png
